In [ ]:
import os
os.environ["OPENAI_API_KEY"] =""

In [ ]:

from __future__ import annotations

import csv
import json
import os
import re
import sys
from pathlib import Path
from typing import List, Optional, Tuple
import sys
!{sys.executable} -m pip install langchain_openai
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, ConfigDict, Field, ValidationError


# ============================================================
# CONFIGURATION
# ============================================================

INPUT_FILE = Path("extracted_text.txt")
OUTPUT_JSON = Path("structured_data.json")
OUTPUT_CSV = Path("flags.csv")

# Initial number of source lines sent to the model per request.
CHUNK_SIZE = 500

# Normal overlap between chunks.
CHUNK_OVERLAP = 75

# If a chunk triggers the content filter, recursively split it
# until it is at most this many lines.
#
# If even a chunk this small is rejected, the source is preserved
# locally rather than sent to the model again.
MIN_CONTENT_FILTER_CHUNK_SIZE = 10

# Model can be changed with:
#
#   export OPENAI_MODEL="gpt-5.4"
#
MODEL_NAME = os.environ.get(
    "OPENAI_MODEL",
    "gpt-5.4",
)


# ============================================================
# IMPORTANT ARCHITECTURE NOTE
# ============================================================
#
# The OpenAI structured-output schema does NOT use arbitrary
# dictionaries such as:
#
#     Dict[str, str]
#
# for credits, sections, performances, etc.
#
# Instead, the LLM returns LISTS of Pydantic objects.
#
# After the LLM response has been validated, Python converts
# those lists into the dictionary structure requested by the
# user.
#
# This avoids the "required ... Extra required key 'credits'"
# structured-output schema error.
#
#
# A second important design principle:
#
#     extracted_text.txt
#
# remains the authoritative source.
#
# The LLM is never allowed to be the only copy of the OCR.
#
# If the content filter rejects some source material, that
# material is preserved verbatim in the output rather than
# discarded.
# ============================================================


# ============================================================
# PYDANTIC MODELS SENT TO THE LLM
# ============================================================


class Credit(BaseModel):
    model_config = ConfigDict(extra="forbid")

    role: str = Field(
        description=(
            "The credited role. Include scope when the credit "
            "only applies to a particular act, section, movement, "
            "scene, or performance. Example: 'composer (Act 1)'."
        )
    )

    names: str = Field(
        description=(
            "Name or names associated with this role. If several "
            "people have the same role, separate their names with "
            "commas."
        )
    )


class Performance(BaseModel):
    model_config = ConfigDict(extra="forbid")

    performance_type: Optional[str] = Field(
        description=(
            "Type of performance, such as song, scene, interlude, "
            "aria, movement, etc. Null if unavailable."
        )
    )

    performance_name: Optional[str] = Field(
        description=(
            "Performance title/name exactly as represented in the "
            "source. Null if unavailable."
        )
    )

    lyrics_transcript: Optional[str] = Field(
        description=(
            "Lyrics or transcript if present. Preserve the OCR "
            "wording rather than correcting or paraphrasing it. "
            "Null if unavailable."
        )
    )

    based_on_tune: Optional[str] = Field(
        description=(
            "Tune on which the song is based, if explicitly stated. "
            "Null if unavailable."
        )
    )

    additional_information: Optional[str] = Field(
        description=(
            "Additional information about this performance that "
            "does not fit another field. Null if unavailable."
        )
    )


class PerformanceEntry(BaseModel):
    model_config = ConfigDict(extra="forbid")

    id: str = Field(
        description=(
            "Unique identifier for this performance within its "
            "section."
        )
    )

    performance: Performance


class Section(BaseModel):
    model_config = ConfigDict(extra="forbid")

    section_type: Optional[str] = Field(
        description=(
            "Type of section, such as act, part, movement, scene, "
            "or full performance. Null if unavailable."
        )
    )

    section_name: Optional[str] = Field(
        description=(
            "Name of the section exactly as represented in the "
            "source. Null if unavailable."
        )
    )

    additional_information: Optional[str] = Field(
        description=(
            "Additional information concerning the section. "
            "Null if unavailable."
        )
    )

    performances: List[PerformanceEntry] = Field(
        description=(
            "Performances contained within this section. Use an "
            "empty list if none are present."
        )
    )


class SectionEntry(BaseModel):
    model_config = ConfigDict(extra="forbid")

    id: str = Field(
        description=(
            "Unique identifier for this section within the event."
        )
    )

    section: Section


class Event(BaseModel):
    model_config = ConfigDict(extra="forbid")

    event_name: Optional[str] = Field(
        description=(
            "Event name exactly as represented in the source. "
            "Null if unavailable."
        )
    )

    event_date: Optional[str] = Field(
        description=(
            "Complete event date in dd/mm/yyyy format if explicitly "
            "available. Do not infer missing components. Null if "
            "unavailable or incomplete."
        )
    )

    event_venue: Optional[str] = Field(
        description=(
            "Event venue exactly as represented in the source. "
            "Null if unavailable."
        )
    )

    event_location: Optional[str] = Field(
        description=(
            "Event location exactly as represented in the source. "
            "Null if unavailable."
        )
    )

    presenter_sponsor: Optional[str] = Field(
        description=(
            "Presenter or sponsor if a single entity is clearly "
            "identified as such. Otherwise place credits in the "
            "credits list. Null if unavailable."
        )
    )

    event_type: Optional[str] = Field(
        description=(
            "Event type, such as concert, opera, recital, lecture, "
            "festival, etc. Use 'non-event' for material that is "
            "completely unrelated to a musical event."
        )
    )

    credits: List[Credit] = Field(
        description=(
            "ALL credits appearing anywhere in the event, including "
            "composer, arranger, conductor, performer, director, "
            "author, translator, etc."
        )
    )

    additional_information: Optional[str] = Field(
        description=(
            "Any event information that does not reasonably fit "
            "elsewhere. Null if unavailable."
        )
    )

    sections: List[SectionEntry] = Field(
        description=(
            "All sections of the event. If there is no meaningful "
            "division, create one section with section_type "
            "'full performance'."
        )
    )


class EventEntry(BaseModel):
    model_config = ConfigDict(extra="forbid")

    id: str = Field(
        description=(
            "Unique identifier for this event within the current "
            "extraction."
        )
    )

    event: Event


class ExtractionFlag(BaseModel):
    model_config = ConfigDict(extra="forbid")

    flag_type: str = Field(
        description=(
            "Short standardized flag type."
        )
    )

    flag_description: str = Field(
        description=(
            "Detailed description of the problem."
        )
    )

    what_was_done: str = Field(
        description=(
            "Description of how the problem was handled."
        )
    )

    source_line_start: int = Field(
        description=(
            "Original source line where the issue begins."
        )
    )

    source_line_end: int = Field(
        description=(
            "Original source line where the issue ends."
        )
    )

    source_excerpt: Optional[str] = Field(
        description=(
            "Short verbatim excerpt identifying the flagged "
            "source material. Null if unavailable."
        )
    )


class ChunkExtraction(BaseModel):
    model_config = ConfigDict(extra="forbid")

    events: List[EventEntry] = Field(
        description=(
            "All events found in the supplied OCR chunk."
        )
    )

    flags: List[ExtractionFlag] = Field(
        description=(
            "All questionable material requiring manual review."
        )
    )


class FinalExtraction(BaseModel):
    model_config = ConfigDict(extra="forbid")

    events: List[EventEntry] = Field(
        description=(
            "All distinct events found in the complete source."
        )
    )

    flags: List[ExtractionFlag] = Field(
        description=(
            "All potential problems requiring manual review."
        )
    )


# ============================================================
# EXTRACTION SYSTEM PROMPT
# ============================================================

EXTRACTION_SYSTEM_PROMPT = r"""
You are a musicologist performing archival OCR transcription.

You are given a portion of a much larger OCR-extracted text file containing
concert programs, opera programs, musical event descriptions, catalogs,
program notes, advertisements, and potentially unrelated material.

Your task is to TRANSCRIBE AND STRUCTURE INFORMATION PRESENT IN THE SOURCE.

You are NOT doing historical research.

============================================================
ABSOLUTE RULES
============================================================

1. Do not consult outside information.

2. Do not infer facts that are not present in the supplied text.

3. Do not silently correct OCR.

4. If a name looks misspelled, preserve the spelling exactly as it appears.

5. If something looks factually impossible, preserve it and create a flag.

6. Do not omit information merely because it does not fit neatly into the
   schema.

7. Put information that does not fit another field into the most appropriate
   additional_information field.

8. If information is completely unrelated to a musical event, create an event
   with event_type "non-event" and put the information into
   additional_information.

9. Every event must contain every event-level field.

10. Use null for unavailable scalar information.

11. Use empty lists when there are no credits, sections, or performances.

12. Preserve lyrics, transcripts, titles, names, dates, and other textual
    material as faithfully as possible.

13. Do not paraphrase material when it can be transcribed.

14. Preserve obvious OCR errors.

15. Do not invent dates, locations, venues, composers, performers,
    organizations, etc.

16. The source may contain several separate documents.

17. Determine document boundaries using evidence in the OCR text, including
    page numbers, headings, titles, dates, venue information, and changes
    in content.

18. A single source document may contain multiple separate events.
    Separate them.

19. A single event may contain multiple acts, parts, movements, scenes, etc.

20. Put performances inside their appropriate sections.

21. If there is no meaningful division, create one section with
    section_type "full performance".

22. ALL credits appearing anywhere in an event must be represented in the
    event's credits list.

23. If a person is credited only for one section or performance, include
    that scope in the role.

24. If multiple people have the same role, use one Credit object and separate
    names using commas.

25. Do not use knowledge about people, works, institutions, dates, or places
    outside the supplied text.

26. When something is genuinely incomprehensible, preserve it as closely as
    possible and create a flag.

27. Do not correct capitalization, punctuation, spelling, or names.

28. If something seems wrong, preserve it and create a flag.

29. If something appears to be an OCR duplicate, create a duplicate-information
    flag.

30. Preserve useful page-number information in additional_information when
    appropriate.

============================================================
DATES
============================================================

Represent dates as dd/mm/yyyy ONLY when the complete date is explicitly
available.

Do not infer missing day, month, or year.

If the complete date cannot be established from the supplied text, use null.

============================================================
FLAGS
============================================================

Use consistent short flag types.

Preferred types:

- potential misspelling
- OCR error
- incomprehensible OCR
- formatting error
- non-musical information
- duplicate information
- ambiguous attribution
- ambiguous event boundary
- incomplete information
- conflicting information

Create a flag whenever manual review would reasonably be useful.

Always provide original source line numbers.

============================================================
CHUNK BOUNDARIES
============================================================

This is only one chunk of a much larger OCR file.

An event may begin before this chunk or continue after it.

Extract only information actually visible in this chunk.

Do not invent information beyond the supplied text.

============================================================
SOURCE PRESERVATION
============================================================

The objective is maximum preservation.

Do not summarize away information.

Do not correct information.

Do not research information.

Do not guess.
"""


# ============================================================
# CONSOLIDATION SYSTEM PROMPT
# ============================================================

CONSOLIDATION_SYSTEM_PROMPT = r"""
You are the final archival editor for a musicological OCR transcription.

You will receive structured fragments extracted from multiple, possibly
overlapping portions of one OCR text file.

Your task is to merge them into one final collection.

============================================================
ABSOLUTE RULES
============================================================

1. Use ONLY information contained in the supplied extraction fragments.

2. Do not use outside knowledge.

3. Do not correct spelling.

4. Do not fix names based on what you believe the correct person is.

5. Preserve OCR wording whenever possible.

6. Do not omit information.

7. If two fragments contain the same information because of chunk overlap,
   deduplicate it.

8. If two fragments contain genuinely different information, preserve both.

9. If the same event occurs across multiple chunks, merge it.

10. If one source document contains multiple distinct events, keep them
    separate.

11. If two events have similar names but insufficient evidence proves that
    they are the same event, keep them separate.

12. Every event must contain every required field.

13. Use null for unavailable scalar information.

14. Use empty lists for empty collections.

15. Every credit appearing anywhere in an event must be represented in
    event-level credits.

16. Section/performance-specific credits must include their scope in the
    role.

17. Preserve lyrics and transcripts.

18. Preserve program notes and miscellaneous information.

19. Preserve non-musical source material as a "non-event".

20. Preserve questionable information and its flags.

21. Do not turn uncertainty into certainty.

22. Do not research anything.

23. Do not silently correct OCR.

24. Do not omit information because it is difficult to classify.

============================================================
DUPLICATE HANDLING
============================================================

The extraction process uses overlapping chunks.

The same event or performance may therefore appear more than once.

Merge clearly identical overlapping material.

Do not duplicate the same performance simply because it appeared in two
overlapping chunks.

If information differs, preserve both and flag the discrepancy if appropriate.

============================================================
EVENT BOUNDARIES
============================================================

Use only source evidence such as:

- event titles
- dates
- venues
- locations
- page numbers
- headings
- program structure
- presenters
- program numbering
- abrupt changes in subject matter

Do not use outside historical knowledge.

============================================================
CREDITS
============================================================

Every credit appearing anywhere in an event must be included.

For example:

composer -> John Smith
arranger -> Mary Jones
composer (Act II) -> Robert Brown

If multiple people have the same role:

violins -> J. Jones, H. Smith

============================================================
FINAL GOAL
============================================================

Produce a complete transcription.

Priority:

1. Preserve information.
2. Preserve OCR wording.
3. Preserve uncertainty.
4. Preserve credits.
5. Preserve structure.
6. Avoid assumptions.
7. Flag questionable material.
"""


# ============================================================
# SOURCE READING
# ============================================================

def read_source(
    path: Path,
) -> List[str]:

    if not path.exists():
        raise FileNotFoundError(
            f"Input file does not exist: {path}"
        )

    text = path.read_text(
        encoding="utf-8",
        errors="replace",
    )

    lines = text.splitlines()

    return [
        f"{line_number:06d} | {line}"
        for line_number, line in enumerate(
            lines,
            start=1,
        )
    ]


# ============================================================
# CHUNK CREATION
# ============================================================

def make_chunks(
    lines: List[str],
    chunk_size: int,
    overlap: int,
) -> List[str]:

    if chunk_size <= overlap:
        raise ValueError(
            "chunk_size must be greater than overlap."
        )

    chunks = []

    start = 0

    while start < len(lines):

        end = min(
            start + chunk_size,
            len(lines),
        )

        chunks.append(
            "\n".join(
                lines[start:end]
            )
        )

        if end >= len(lines):
            break

        start = end - overlap

    return chunks


# ============================================================
# MODEL CREATION
# ============================================================

def make_model() -> ChatOpenAI:

    if not os.environ.get(
        "OPENAI_API_KEY"
    ):
        raise RuntimeError(
            "OPENAI_API_KEY environment variable is not set."
        )

    return ChatOpenAI(
        model=MODEL_NAME,
        temperature=0,
    )


# ============================================================
# CONTENT FILTER DETECTION
# ============================================================

def is_content_filter_error(
    error: Exception,
) -> bool:

    text = str(error).lower()

    indicators = [
        "contentfilterfinishreasonerror",
        "content filter",
        "content_filter",
        "contentfilter",
        "finish_reason",
        "finish reason",
    ]

    return any(
        indicator in text
        for indicator in indicators
    )


# ============================================================
# GET SOURCE LINE RANGE
# ============================================================

def get_line_range(
    chunk: str,
) -> Tuple[int, int]:

    lines = chunk.splitlines()

    if not lines:
        return (0, 0)

    numbers = []

    for line in lines:

        match = re.match(
            r"\s*(\d+)\s*\|",
            line,
        )

        if match:
            numbers.append(
                int(match.group(1))
            )

    if not numbers:
        return (0, 0)

    return (
        min(numbers),
        max(numbers),
    )


# ============================================================
# CONTENT-FILTER FALLBACK EVENT
# ============================================================

def create_content_filter_fallback(
    chunk: str,
) -> Tuple[EventEntry, ExtractionFlag]:

    start_line, end_line = (
        get_line_range(chunk)
    )

    # Preserve the ORIGINAL OCR exactly as supplied to the
    # model. This is critical for your no-information-loss
    # requirement.
    preserved_text = (
        "[SOURCE TEXT PRESERVED DUE TO "
        "CONTENT-FILTER REJECTION]\n\n"
        + chunk
    )

    event = Event(
        event_name=None,
        event_date=None,
        event_venue=None,
        event_location=None,
        presenter_sponsor=None,
        event_type="non-event",
        credits=[],
        additional_information=preserved_text,
        sections=[],
    )

    event_entry = EventEntry(
        id=(
            f"content_filter_{start_line}_"
            f"{end_line}"
        ),
        event=event,
    )

    flag = ExtractionFlag(
        flag_type="content filter",
        flag_description=(
            "The OCR source lines were rejected by the model "
            "content filter and therefore could not be structurally "
            "transcribed by the language model."
        ),
        what_was_done=(
            "The original OCR text was preserved verbatim in the "
            "additional_information field of a non-event record. "
            "No attempt was made to rewrite, correct, summarize, "
            "or omit the rejected material."
        ),
        source_line_start=start_line,
        source_line_end=end_line,
        source_excerpt=chunk[:1000],
    )

    return event_entry, flag


# ============================================================
# SINGLE CHUNK EXTRACTION
# ============================================================

def extract_chunk_once(
    model: ChatOpenAI,
    chunk: str,
    label: str,
) -> ChunkExtraction:

    print(
        f"  Processing {label}...",
        flush=True,
    )

    structured_model = model.with_structured_output(
        ChunkExtraction,
        method="json_schema",
    )

    prompt = f"""
{EXTRACTION_SYSTEM_PROMPT}

============================================================
SOURCE CHUNK
============================================================

{chunk}

============================================================
END SOURCE CHUNK
============================================================

Extract every piece of information that can be recovered.

Preserve source wording.

Do not correct OCR.

Do not use outside information.
"""

    result = structured_model.invoke(
        prompt
    )

    return ChunkExtraction.model_validate(
        result
    )


# ============================================================
# RECURSIVE CONTENT-FILTER-SAFE EXTRACTION
# ============================================================

def extract_chunk_resilient(
    model: ChatOpenAI,
    chunk: str,
    label: str,
    depth: int = 0,
) -> Tuple[
    List[EventEntry],
    List[ExtractionFlag],
]:

    lines = chunk.splitlines()

    # --------------------------------------------------------
    # Empty chunk
    # --------------------------------------------------------

    if not lines:
        return [], []

    # --------------------------------------------------------
    # Try normal extraction
    # --------------------------------------------------------

    try:

        result = extract_chunk_once(
            model=model,
            chunk=chunk,
            label=label,
        )

        return (
            result.events,
            result.flags,
        )

    except Exception as exc:

        # ----------------------------------------------------
        # Non-content-filter error
        # ----------------------------------------------------

        if not is_content_filter_error(
            exc
        ):

            print(
                f"  ERROR in {label}: {exc}",
                file=sys.stderr,
            )

            raise

        print(
            f"  Content filter rejected {label}.",
            flush=True,
        )

        # ----------------------------------------------------
        # If already sufficiently small, preserve it locally.
        # ----------------------------------------------------

        if len(lines) <= MIN_CONTENT_FILTER_CHUNK_SIZE:

            print(
                f"  Preserving rejected source "
                f"locally ({len(lines)} lines).",
                flush=True,
            )

            event, flag = (
                create_content_filter_fallback(
                    chunk
                )
            )

            return (
                [event],
                [flag],
            )

        # ----------------------------------------------------
        # Otherwise split the chunk into two pieces.
        # ----------------------------------------------------

        midpoint = len(lines) // 2

        left_lines = lines[
            :midpoint
        ]

        right_lines = lines[
            midpoint:
        ]

        left_chunk = "\n".join(
            left_lines
        )

        right_chunk = "\n".join(
            right_lines
        )

        left_events, left_flags = (
            extract_chunk_resilient(
                model=model,
                chunk=left_chunk,
                label=f"{label}.A",
                depth=depth + 1,
            )
        )

        right_events, right_flags = (
            extract_chunk_resilient(
                model=model,
                chunk=right_chunk,
                label=f"{label}.B",
                depth=depth + 1,
            )
        )

        return (
            left_events + right_events,
            left_flags + right_flags,
        )


# ============================================================
# CLEAN IDS
# ============================================================

def clean_id(
    value: Optional[str],
) -> str:

    if not value:
        return "unnamed"

    value = re.sub(
        r"[^A-Za-z0-9_-]+",
        "_",
        value,
    )

    value = value.strip("_")

    return value or "unnamed"


def unique_id(
    existing: set[str],
    proposed: str,
    prefix: str,
) -> str:

    proposed = clean_id(
        proposed
    )

    if not proposed:
        proposed = prefix

    candidate = proposed

    number = 2

    while candidate in existing:

        candidate = (
            f"{proposed}_{number}"
        )

        number += 1

    existing.add(
        candidate
    )

    return candidate


# ============================================================
# CONVERT EVENT TO FINAL DICTIONARY FORMAT
# ============================================================

def event_to_final_dict(
    event: Event,
) -> dict:

    # --------------------------------------------------------
    # Credits
    # --------------------------------------------------------

    credits = {}

    for credit in event.credits:

        role = credit.role.strip()

        names = credit.names.strip()

        if role not in credits:

            credits[role] = names

        else:

            existing = [
                x.strip()
                for x in credits[role].split(",")
            ]

            for name in names.split(","):

                name = name.strip()

                if (
                    name
                    and name not in existing
                ):
                    existing.append(
                        name
                    )

            credits[role] = (
                ", ".join(existing)
            )

    # --------------------------------------------------------
    # Sections
    # --------------------------------------------------------

    sections = {}

    section_ids = set()

    for section_entry in event.sections:

        section = (
            section_entry.section
        )

        section_id = unique_id(
            section_ids,
            section_entry.id,
            "section",
        )

        performances = {}

        performance_ids = set()

        for performance_entry in (
            section.performances
        ):

            performance = (
                performance_entry.performance
            )

            performance_id = unique_id(
                performance_ids,
                performance_entry.id,
                "performance",
            )

            performances[
                performance_id
            ] = {
                "performance_type":
                    performance.performance_type,

                "performance_name":
                    performance.performance_name,

                "lyrics_transcript":
                    performance.lyrics_transcript,

                "based_on_tune":
                    performance.based_on_tune,

                "additional_information":
                    performance.additional_information,
            }

        sections[
            section_id
        ] = {
            "section_type":
                section.section_type,

            "section_name":
                section.section_name,

            "additional_information":
                section.additional_information,

            "performances":
                performances,
        }

    # --------------------------------------------------------
    # Event
    # --------------------------------------------------------

    return {
        "event_name":
            event.event_name,

        "event_date":
            event.event_date,

        "event_venue":
            event.event_venue,

        "event_location":
            event.event_location,

        "presenter_sponsor":
            event.presenter_sponsor,

        "event_type":
            event.event_type,

        "credits":
            credits,

        "additional_information":
            event.additional_information,

        "sections":
            sections,
    }


# ============================================================
# CONSOLIDATION
# ============================================================

def consolidate(
    model: ChatOpenAI,
    fragments: List[ChunkExtraction],
    fallback_events: List[EventEntry],
    fallback_flags: List[ExtractionFlag],
) -> FinalExtraction:

    print(
        "\nPreparing extraction fragments..."
    )

    events = []

    flags = []

    # Regular extraction results.
    for fragment in fragments:

        events.extend(
            fragment.events
        )

        flags.extend(
            fragment.flags
        )

    # Content-filter-preserved events.
    events.extend(
        fallback_events
    )

    flags.extend(
        fallback_flags
    )

    # --------------------------------------------------------
    # Nothing to consolidate
    # --------------------------------------------------------

    if not events:

        return FinalExtraction(
            events=[],
            flags=flags,
        )

    # --------------------------------------------------------
    # Serialize
    # --------------------------------------------------------

    extraction_json = json.dumps(
        {
            "events": [
                x.model_dump(
                    mode="json"
                )
                for x in events
            ],

            "flags": [
                x.model_dump(
                    mode="json"
                )
                for x in flags
            ],
        },
        ensure_ascii=False,
        indent=2,
    )

    # --------------------------------------------------------
    # IMPORTANT:
    #
    # If the consolidated fragments themselves are too large,
    # a single consolidation call can hit the context window.
    #
    # This code therefore checks the approximate character
    # count and skips LLM consolidation when there is nothing
    # reasonably safe to consolidate.
    #
    # In that situation, the already-Pydantic-validated events
    # are retained.
    # --------------------------------------------------------

    MAX_CONSOLIDATION_CHARS = 600_000

    if len(extraction_json) > (
        MAX_CONSOLIDATION_CHARS
    ):

        print(
            "\nExtraction is too large for a single "
            "consolidation request."
        )

        print(
            "Retaining individually validated events "
            "without LLM consolidation."
        )

        return FinalExtraction(
            events=events,
            flags=flags,
        )

    # --------------------------------------------------------
    # Consolidate
    # --------------------------------------------------------

    print(
        "\nConsolidating extracted events..."
    )

    structured_model = model.with_structured_output(
        FinalExtraction,
        method="json_schema",
    )

    prompt = f"""
{CONSOLIDATION_SYSTEM_PROMPT}

============================================================
EXTRACTED MATERIAL
============================================================

{extraction_json}

============================================================
END EXTRACTED MATERIAL
============================================================

Consolidate these fragments into one final extraction.

Do not omit information.

Do not correct OCR.

Do not use outside knowledge.

Merge only material that is clearly duplicated because of chunk
overlap.

Keep genuinely distinct events separate.
"""

    try:

        result = structured_model.invoke(
            prompt
        )

        final = FinalExtraction.model_validate(
            result
        )

        return final

    except Exception as exc:

        # ----------------------------------------------------
        # Content filter during consolidation
        #
        # This can happen because the serialized fragments
        # contain source text that the initial extraction
        # successfully handled but the consolidation request
        # rejects.
        #
        # In that case, do NOT lose the already extracted data.
        # ----------------------------------------------------

        if is_content_filter_error(
            exc
        ):

            print(
                "\nContent filter rejected the "
                "consolidation request.",
                file=sys.stderr,
            )

            print(
                "Using already validated extraction "
                "fragments without consolidation.",
                file=sys.stderr,
            )

            flags.append(
                ExtractionFlag(
                    flag_type="content filter",
                    flag_description=(
                        "The consolidation request was rejected "
                        "by the model content filter."
                    ),
                    what_was_done=(
                        "The individually extracted and Pydantic-"
                        "validated event records were retained "
                        "without performing the final LLM "
                        "consolidation step."
                    ),
                    source_line_start=0,
                    source_line_end=0,
                    source_excerpt=None,
                )
            )

            return FinalExtraction(
                events=events,
                flags=flags,
            )

        raise


# ============================================================
# FINAL JSON VALIDATION MODELS
# ============================================================


class FinalPerformance(BaseModel):
    model_config = ConfigDict(extra="forbid")

    performance_type: Optional[str]
    performance_name: Optional[str]
    lyrics_transcript: Optional[str]
    based_on_tune: Optional[str]
    additional_information: Optional[str]


class FinalSection(BaseModel):
    model_config = ConfigDict(extra="forbid")

    section_type: Optional[str]
    section_name: Optional[str]
    additional_information: Optional[str]

    performances: dict[
        str,
        FinalPerformance,
    ]


class FinalEvent(BaseModel):
    model_config = ConfigDict(extra="forbid")

    event_name: Optional[str]
    event_date: Optional[str]
    event_venue: Optional[str]
    event_location: Optional[str]
    presenter_sponsor: Optional[str]
    event_type: Optional[str]

    credits: dict[
        str,
        str,
    ]

    additional_information: Optional[str]

    sections: dict[
        str,
        FinalSection,
    ]


class FinalJSON(BaseModel):
    model_config = ConfigDict(extra="forbid")

    Events: dict[
        str,
        FinalEvent,
    ]


# ============================================================
# CONVERT TO FINAL JSON
# ============================================================

def convert_to_final_json(
    extraction: FinalExtraction,
) -> dict:

    output = {
        "Events": {}
    }

    event_ids = set()

    for event_entry in extraction.events:

        event_id = unique_id(
            event_ids,
            event_entry.id,
            "event",
        )

        output["Events"][
            event_id
        ] = event_to_final_dict(
            event_entry.event
        )

    return output


# ============================================================
# FLAG DEDUPLICATION
# ============================================================

def deduplicate_flags(
    flags: List[ExtractionFlag],
) -> List[ExtractionFlag]:

    seen = set()

    result = []

    for flag in flags:

        key = (
            flag.flag_type.lower().strip(),
            flag.source_line_start,
            flag.source_line_end,
            flag.flag_description.strip(),
        )

        if key in seen:
            continue

        seen.add(key)

        result.append(
            flag
        )

    return result


# ============================================================
# FIND JSON LOCATION
# ============================================================

def find_json_location(
    json_text: str,
    excerpt: Optional[str],
) -> str:

    if not excerpt:
        return "not located"

    excerpt = excerpt.strip()

    if not excerpt:
        return "not located"

    json_lines = (
        json_text.splitlines()
    )

    candidates = [
        excerpt
    ]

    words = excerpt.split()

    if len(words) >= 12:
        candidates.append(
            " ".join(words[:12])
        )

    if len(words) >= 8:
        candidates.append(
            " ".join(words[:8])
        )

    if len(words) >= 5:
        candidates.append(
            " ".join(words[:5])
        )

    for candidate in candidates:

        if not candidate:
            continue

        for line_number, line in enumerate(
            json_lines,
            start=1,
        ):

            if candidate in line:

                return str(
                    line_number
                )

    return "not located"


# ============================================================
# WRITE JSON
# ============================================================

def write_json(
    data: dict,
) -> str:

    json_text = json.dumps(
        data,
        ensure_ascii=False,
        indent=2,
    )

    OUTPUT_JSON.write_text(
        json_text + "\n",
        encoding="utf-8",
    )

    return json_text


# ============================================================
# WRITE CSV
# ============================================================

def write_csv(
    flags: List[ExtractionFlag],
    json_text: str,
) -> None:

    flags = deduplicate_flags(
        flags
    )

    with OUTPUT_CSV.open(
        "w",
        encoding="utf-8",
        newline="",
    ) as f:

        writer = csv.writer(
            f
        )

        writer.writerow(
            [
                "Flag Type",
                "Flag description",
                "What was done",
                "Where found",
                "Where placed",
            ]
        )

        for flag in flags:

            source_location = (
                f"{flag.source_line_start}-"
                f"{flag.source_line_end}"
            )

            json_location = (
                find_json_location(
                    json_text,
                    flag.source_excerpt,
                )
            )

            writer.writerow(
                [
                    flag.flag_type,
                    flag.flag_description,
                    flag.what_was_done,
                    source_location,
                    json_location,
                ]
            )


# ============================================================
# MAIN
# ============================================================

def main() -> None:

    print("=" * 70)
    print(
        "MUSICOLOGICAL OCR EVENT EXTRACTION"
    )
    print("=" * 70)

    # --------------------------------------------------------
    # Read source
    # --------------------------------------------------------

    print(
        "\nReading extracted_text.txt..."
    )

    source_lines = read_source(
        INPUT_FILE
    )

    print(
        f"Source lines: "
        f"{len(source_lines):,}"
    )

    # --------------------------------------------------------
    # Create chunks
    # --------------------------------------------------------

    chunks = make_chunks(
        source_lines,
        CHUNK_SIZE,
        CHUNK_OVERLAP,
    )

    print(
        f"Initial chunks: "
        f"{len(chunks):,}"
    )

    # --------------------------------------------------------
    # Create model
    # --------------------------------------------------------

    print(
        f"Model: {MODEL_NAME}"
    )

    model = make_model()

    # --------------------------------------------------------
    # Extract every chunk
    # --------------------------------------------------------

    fragments: List[
        ChunkExtraction
    ] = []

    fallback_events: List[
        EventEntry
    ] = []

    fallback_flags: List[
        ExtractionFlag
    ] = []

    for chunk_number, chunk in enumerate(
        chunks,
        start=1,
    ):

        print()
        print(
            f"========== CHUNK "
            f"{chunk_number}/{len(chunks)} =========="
        )

        try:

            events, flags = (
                extract_chunk_resilient(
                    model=model,
                    chunk=chunk,
                    label=(
                        f"chunk_{chunk_number}"
                    ),
                )
            )

            # Separate content-filter fallback
            # events from normal extraction.
            #
            # Fallback events are already valid EventEntry
            # objects, but keeping them separate makes it
            # possible to preserve them even if consolidation
            # later fails.
            for event in events:

                if (
                    event.event.event_type
                    == "non-event"
                    and event.event.additional_information
                    and event.event.additional_information.startswith(
                        "[SOURCE TEXT PRESERVED DUE TO "
                        "CONTENT-FILTER REJECTION]"
                    )
                ):

                    fallback_events.append(
                        event
                    )

                else:

                    # Put ordinary events into a
                    # ChunkExtraction object.
                    #
                    # We use one fragment per source chunk.
                    pass

            # Build a fragment from all ordinary events.
            normal_events = [
                event
                for event in events
                if not (
                    event.event.event_type
                    == "non-event"
                    and event.event.additional_information
                    and event.event.additional_information.startswith(
                        "[SOURCE TEXT PRESERVED DUE TO "
                        "CONTENT-FILTER REJECTION]"
                    )
                )
            ]

            fragments.append(
                ChunkExtraction(
                    events=normal_events,
                    flags=flags,
                )
            )

            # Content-filter flags from recursive fallback
            # are included here.
            for flag in flags:

                if (
                    flag.flag_type
                    == "content filter"
                ):

                    fallback_flags.append(
                        flag
                    )

        except Exception as exc:

            print(
                "\nFatal extraction error:",
                file=sys.stderr,
            )

            print(
                exc,
                file=sys.stderr,
            )

            raise

    # --------------------------------------------------------
    # Consolidate
    # --------------------------------------------------------

    final_extraction = consolidate(
        model=model,
        fragments=fragments,
        fallback_events=fallback_events,
        fallback_flags=fallback_flags,
    )

    # --------------------------------------------------------
    # Final Pydantic validation of LLM result
    # --------------------------------------------------------

    print(
        "\nRunning final Pydantic validation..."
    )

    final_extraction = (
        FinalExtraction.model_validate(
            final_extraction.model_dump(
                mode="python"
            )
        )
    )

    # --------------------------------------------------------
    # Convert to requested dictionary structure
    # --------------------------------------------------------

    print(
        "Converting to final dictionary structure..."
    )

    final_json = convert_to_final_json(
        final_extraction
    )

    # --------------------------------------------------------
    # Validate final JSON shape
    # --------------------------------------------------------

    print(
        "Validating final JSON structure..."
    )

    FinalJSON.model_validate(
        final_json
    )

    print(
        "Final JSON validation successful."
    )

    # --------------------------------------------------------
    # Write JSON
    # --------------------------------------------------------

    print(
        f"Writing {OUTPUT_JSON}..."
    )

    json_text = write_json(
        final_json
    )

    # --------------------------------------------------------
    # Write CSV
    # --------------------------------------------------------

    print(
        f"Writing {OUTPUT_CSV}..."
    )

    write_csv(
        final_extraction.flags,
        json_text,
    )

    # --------------------------------------------------------
    # Summary
    # --------------------------------------------------------

    flags = deduplicate_flags(
        final_extraction.flags
    )

    content_filter_count = sum(
        1
        for flag in flags
        if flag.flag_type
        == "content filter"
    )

    print()
    print("=" * 70)
    print("COMPLETE")
    print("=" * 70)

    print(
        f"Events: "
        f"{len(final_json['Events'])}"
    )

    print(
        f"Flags: "
        f"{len(flags)}"
    )

    print(
        f"Content-filtered regions: "
        f"{content_filter_count}"
    )

    print(
        f"JSON: "
        f"{OUTPUT_JSON.resolve()}"
    )

    print(
        f"CSV:  "
        f"{OUTPUT_CSV.resolve()}"
    )

    print("=" * 70)


# ============================================================
# ENTRY POINT
# ============================================================

if __name__ == "__main__":
    main()
